In [0]:
%pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 105.3 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# 💰 Revenue Modeling for SaaS FinTech Funnel

#This notebook builds a **decision layer** on top of SQL analytics.

### Goals:
#Quantify revenue drivers
#Simulate product improvements
#Validate A/B tests
#Estimate customer value (CLTV)
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from statsmodels.stats.proportion import proportions_ztest

In [0]:
# Load daily metrics from raw data (Databricks Volumes)
df = spark.read.format("csv") \
    .option("header", "true") \
    .load("/Volumes/workspace/default/rawdata/data/daily_metrics.csv")

df = df.toPandas()

df.head()

,date,total_orders,workflow_entries,login_attempts,login_successes,completed_orders,daily_revenue,conversion_rate
0,2024-01-01,49,20,16,5,4,8411.32,0.0816
1,2024-01-02,61,29,22,5,4,12465.36,0.0656
2,2024-01-03,49,26,21,5,2,6246.58,0.0408
3,2024-01-04,56,28,15,2,0,0.0,0.0
4,2024-01-05,68,34,17,4,3,8161.56,0.0441


In [0]:
# Convert to numeric
cols = [
    'total_orders',
    'workflow_entries',
    'login_attempts',
    'login_successes',
    'completed_orders',
    'daily_revenue'
]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.fillna(0)

In [0]:
# Create key business drivers
df['login_success_rate'] = df['login_successes'] / df['login_attempts']
df['interaction_rate'] = df['workflow_entries'] / df['total_orders']

df = df.replace([np.inf, -np.inf], 0)

df[['login_success_rate', 'interaction_rate']].describe()

,login_success_rate,interaction_rate
count,181.000000,181.000000
mean,0.231781,0.510702
std,0.100477,0.069404
min,0.000000,0.325581
25%,0.157895,0.470588
50%,0.214286,0.508475
75%,0.300000,0.553571
max,0.555556,0.666667


In [0]:
X = df[['login_success_rate', 'interaction_rate']]
y = df['daily_revenue']

model = LinearRegression()
model.fit(X, y)

print("=== Model Coefficients ===")
print(f"Login Success Impact: {model.coef_[0]:.2f}")
print(f"Interaction Rate Impact: {model.coef_[1]:.2f}")
print(f"Intercept: {model.intercept_:.2f}")

=== Model Coefficients ===
Login Success Impact: 34094.80
Interaction Rate Impact: 14925.01
Intercept: -7869.20


In [0]:
login_impact = model.coef_[0]
interaction_impact = model.coef_[1]

print(f"+1% Login Success → ${login_impact/100:.2f} revenue/day")
print(f"+1% Interaction Rate → ${interaction_impact/100:.2f} revenue/day")

+1% Login Success → $340.95 revenue/day
+1% Interaction Rate → $149.25 revenue/day


In [0]:
def simulate_revenue(login_rate, interaction_rate):
    return model.predict([[login_rate, interaction_rate]])[0]

# Current baseline (from your SQL insights)
baseline = simulate_revenue(0.24, 0.51)

# Improved scenario
improved = simulate_revenue(0.30, 0.60)

print("=== Scenario Simulation ===")
print(f"Baseline Revenue: ${baseline:.2f}")
print(f"Improved Revenue: ${improved:.2f}")
print(f"Revenue Lift: ${improved - baseline:.2f}")

=== Scenario Simulation ===
Baseline Revenue: $7925.30
Improved Revenue: $11314.24
Revenue Lift: $3388.94


/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [0]:
# Example experiment data
success = [382, 289]   # variant B, control
users = [1200, 1200]

stat, pval = proportions_ztest(success, users)

print("=== A/B Test Results ===")
print(f"Z-statistic: {stat:.4f}")
print(f"P-value: {pval:.4f}")

if pval < 0.05:
    print("✅ Statistically significant (Ship the feature)")
else:
    print("❌ Not statistically significant")

=== A/B Test Results ===
Z-statistic: 4.2299
P-value: 0.0000
✅ Statistically significant (Ship the feature)


In [0]:
orders = spark.read.format("csv") \
    .option("header", "true") \
    .load("/Volumes/workspace/default/rawdata/data/orders.csv")

orders = orders.toPandas()

# Convert numeric
orders['order_value'] = pd.to_numeric(orders['order_value'], errors='coerce')

orders.head()

,order_id,client_id,created_at,status,entered_workflow,attempted_login,login_success,order_complete,order_value
0,ORD-000001,CLT-066,2024-05-08T21:46:59,initiated_only,0,0,0,0,NaN
1,ORD-000002,CLT-031,2024-05-10T15:27:18,workflow_abandoned,1,0,0,0,NaN
2,ORD-000003,CLT-006,2024-02-17T13:02:56,initiated_only,0,0,0,0,NaN
3,ORD-000004,CLT-010,2024-03-29T10:23:35,completed,1,1,1,1,2573.01
4,ORD-000005,CLT-032,2024-01-29T15:43:56,initiated_only,0,0,0,0,NaN


In [0]:
cltv = orders.groupby('client_id').agg({
    'order_value': 'sum',
    'order_id': 'count'
}).rename(columns={
    'order_value': 'total_revenue',
    'order_id': 'total_orders'
})

cltv['avg_order_value'] = cltv['total_revenue'] / cltv['total_orders']
cltv['cltv'] = cltv['total_revenue']

cltv = cltv.sort_values('cltv', ascending=False)

cltv.head(10)

,total_revenue,total_orders,avg_order_value,cltv
client_id,,,,
CLT-034,38738.44,145,267.161655,38738.44
CLT-065,35272.52,129,273.430388,35272.52
CLT-050,34194.96,135,253.296000,34194.96
CLT-030,32606.41,131,248.903893,32606.41
CLT-009,31937.12,121,263.943140,31937.12
CLT-054,31342.76,141,222.289078,31342.76
CLT-020,31152.88,141,220.942411,31152.88
CLT-039,28871.65,138,209.214855,28871.65
CLT-004,27988.35,127,220.380709,27988.35


In [0]:
def segment_user(revenue):
    if revenue >= 10000:
        return "VIP"
    elif revenue >= 5000:
        return "High Value"
    elif revenue >= 1000:
        return "Medium Value"
    else:
        return "Low Value"

cltv['segment'] = cltv['cltv'].apply(segment_user)

print("=== User Segmentation ===")
print(cltv['segment'].value_counts())

=== User Segmentation ===
segment
VIP             65
High Value      12
Medium Value     3
Name: count, dtype: int64


# 📊 Key Insights & Recommendations

## Revenue Drivers Model

**Impact per 1% improvement:**
* **Login Success Rate:** $340.95/day (2.3x more valuable)
* **Interaction Rate:** $149.25/day

**Model Coefficients:**
* Login Success Impact: $34,095
* Interaction Rate Impact: $14,925
* Intercept: -$7,869

---

## Business Metrics (181 days analyzed)

**Login Success Rate:**
* Average: 23.2%
* Range: 0% to 55.6%
* 50th percentile: 21.4%
* ⚠️ **CRITICAL BOTTLENECK** - Significant room for improvement

**Interaction Rate:**
* Average: 51.1%
* Range: 32.6% to 66.7%
* 50th percentile: 50.8%

---

## Revenue Opportunity

**Current Baseline:** $7,925/day
* Login rate: 24%
* Interaction rate: 51%

**Improved Scenario:** $11,314/day
* Login rate: 30% (+6 points)
* Interaction rate: 60% (+9 points)

**Potential Revenue Lift:** $3,389/day (42.7% increase)
* Annualized: ~$1.24M additional revenue

---

## A/B Test Results

* **Z-statistic:** 4.23
* **P-value:** 0.0000
* **Conclusion:** ✅ Statistically significant (p < 0.05)
* **Recommendation:** Ship the feature

---

## Customer Segmentation

**Distribution:**
* VIP (≥$10K): 65 customers
* High Value ($5K-$10K): 12 customers
* Medium Value ($1K-$5K): 3 customers

**Top 5 Customers by CLTV:**
1. CLT-034: $38,738 (145 orders)
2. CLT-065: $35,273 (129 orders)
3. CLT-050: $34,195 (135 orders)
4. CLT-030: $32,606 (131 orders)
5. CLT-009: $31,937 (121 orders)

**VIP Segment Metrics:**
* 65 customers drive majority of revenue
* Average orders per VIP: ~130 orders
* Average order value: ~$250-270

---

## Strategic Recommendations

### 1. 🚨 URGENT: Fix Login Flow (Highest ROI)
* Current 23% success rate indicates severe authentication issues
* Each 1% improvement = $341/day additional revenue
* Target: Increase from 23% to 30%+ (potential $2,400/day lift)
* Actions:
  * Audit authentication errors
  * Simplify login UX
  * Add password reset improvements
  * Monitor error codes and failure reasons

### 2. 💎 VIP Customer Strategy
* 65 customers represent the core revenue base
* Implement dedicated success team
* Proactive engagement and support
* Early access to new features
* Risk: High concentration - monitor churn closely

### 3. 📈 Conversion Optimization
* Focus on workflow interaction improvements
* Target: Increase from 51% to 60% (+9 points)
* Test simplified workflows
* A/B test onboarding improvements

### 4. 🔬 Continue Experimentation
* Current A/B test shows statistical significance
* Ship winning variants quickly
* Test more aggressive login improvements
* Measure impact on revenue directly

---

## KPIs to Monitor

1. **Daily login success rate** (target: >30%)
2. **Daily interaction rate** (target: >55%)
3. **VIP customer health** (order frequency, last order date)
4. **Daily revenue trend** (7-day rolling average)
5. **Conversion funnel drop-offs** (especially login stage)